# Data Loading

## Imports

In [ ]:
from dotenv import load_dotenv
load_dotenv()
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from constants import AIRPORT_LIMIT_LIST, AIRLINE_LIMIT_LIST, DROP_COLS
from loader import LoaderStorage


In [ ]:
min_features = ["FlightDate","CRSDepTime","CRSArrTime","DepTime","Origin","Dest","Reporting_Airline","DepDelayMinutes","ArrDelayMinutes","CarrierDelay","NASDelay","LateAircraftDelay"]

In [ ]:
storage = LoaderStorage(
  root="s3://data-mining/"
)


# the airlines are already filterd to the ones only that we use.
Source_folder = "data/raw/"
Destination_folder = "data/interim/"
output_file = "1_year_data_more.parquet"

In [ ]:
df = storage.read_csv(f"{Source_folder}On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2013_1.csv")

for idx,col in enumerate(df.columns):
    # print(idx, col)
    if idx > 61:
        DROP_COLS.append(col)

df = df.drop(columns=DROP_COLS)
display(df.head())
for idx,col in enumerate(df.columns):
    print(idx, col)
    # if idx > 60:
    #     drop_cols.append(col)

In [ ]:
# filter the dataframe to only include rows where the origin and dest airports are in the limit list
display(df.head())

In [ ]:
# 1. Define only the columns you actually need
keep_cols = [col for col in df.columns if col not in DROP_COLS]

# convert the list to a dict with the data types for the columns we want to keep based on df
dtype_dict = {col: df[col].dtype for col in keep_cols}
display(dtype_dict)

# 2. Use a list to store dataframes temporarily
year_start = 13
year_end = 15
df_list = []
min_features_df_list = []
for j in range(year_start, year_end + 1):
    for i in range(1, 13):
        df_name = f"On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_20{j}_{i}.csv"
        file_path = Source_folder + df_name
        try:
            # 3. Only read the necessary columns (Saves massive RAM)
            # 4. Force C engine to avoid Arrow mask issues
            df_part = storage.read_csv(
                file_path,
                usecols=keep_cols,
                dtype=dtype_dict,
                engine="c",
                low_memory=False
            )
            df_copy = df_part[min_features].copy()
            print(f"Processing Year: 20{j:02d}, Month: {i}")

            # 5. Filter immediately (Saves RAM for the final concat)
            mask = (
                df_part["Reporting_Airline"].isin(AIRLINE_LIMIT_LIST) &
                (df_part["Origin"].isin(AIRPORT_LIMIT_LIST) |
                df_part["Dest"].isin(AIRPORT_LIMIT_LIST))
            )
            df_part = df_part[mask]

            df_list.append(df_part)
            min_features_df_list.append(df_copy)
        except FileNotFoundError:
            print(f"Skipping: {df_name} not found.")

# 6. Concatenate everything once at the end
if df_list:
    df_all = pd.concat(df_list, ignore_index=True)
else:
    df_all = pd.DataFrame(columns=keep_cols)

# 7. Final cleanup
import gc
del df_list
# 8. Do the same for the min features dataframe
if min_features_df_list:
    min_features_df_all = pd.concat(min_features_df_list, ignore_index=True)
else:    
    min_features_df_all = pd.DataFrame(columns=min_features)
# save the min features dataframe to a parquet file
del min_features_df_list
gc.collect()

In [ ]:
traffic = f"s3://data-mining/{Destination_folder}min_features.parquet"
min_features_df_all.to_parquet(traffic, index=False, storage_options=storage.storage_options)
del min_features_df_all
gc.collect()

In [ ]:
df_all = df_all[~((df_all['ArrTime'].isna()) & (df_all['Cancelled'] == 0) & (df_all['Diverted'] == 0))]

In [ ]:
# colums with times of day
time_cols =["CRSDepTime","DepTime","WheelsOff","WheelsOn","CRSArrTime","ArrTime"]

cancelled_mask = (df_all['Cancelled'] == 1)
diverted_mask = (df_all['Diverted'] == 1)
cancelled_or_diverted_mask = cancelled_mask | diverted_mask
for col in time_cols:
    missing_values = df_all[col].isna()
    df_all[col] = df_all[col].fillna(0).astype(int)
    # convert all 2400 values to 0000, because 2400 is not a valid time, but 0000 is
    df_all[col] = df_all[col].replace(2400, 0)
    df_all[col] = df_all[col].apply(lambda x: f"{x:04d}")
    df_all[col] = pd.to_datetime(df_all[col], format="%H%M", errors='coerce').dt.time
    # make all values NaT if the flight was canncelled
    df_all.loc[cancelled_mask & missing_values, col] = pd.NaT
df_all.loc[diverted_mask, 'WheelsOn'] = pd.NaT
df_all.loc[diverted_mask, 'ArrTime'] = pd.NaT

display(df_all[time_cols].head())

In [ ]:
num_rows = len(df_all)
dataset_completed_flights = df_all[(df_all['Cancelled'] == 0) & (df_all['Diverted'] == 0)]
for column in dataset_completed_flights.columns:
    if dataset_completed_flights[column].isna().sum() > 0:
        # calcuate the percentage of nan values in the column
        percentage_nan = dataset_completed_flights[column].isna().sum() / num_rows * 100
        print(f"Column {column} has {dataset_completed_flights[column].isna().sum()} nan values ({percentage_nan:.2f}%)")
# drop all rows where the arrival delay is Nan, even if the flight was not cancelled or diverted, because these are not useful for the model building


In [ ]:
# load the airport dataset
airports = storage.read_csv('data/external/airports_with_runway_info.csv')
# only keep the iata_code and the Timezone
airports = airports[['iata_code','TZ']]
display(airports.head())

# get the unique values for TZ
print(airports['TZ'].unique())

In [ ]:
# we want to calculate the turaround time of an aircraft
# to make this easier for now, we will reduce the dataset to only the relevant columns
dataset = df_all

# drop all flights, where the DEST or ORIGIN is "PFN"
dataset = dataset[(dataset['Origin'] != 'PFN') & (dataset['Dest'] != 'PFN')]
# we join the aiports dataset to get the timezone of the departure and arrival airports
dataset = dataset.merge(airports, left_on='Origin', right_on='iata_code', how='left')
dataset = dataset.merge(airports, left_on='Dest', right_on='iata_code', how='left', suffixes=('_Origin', '_Dest'))
dataset = dataset.drop(columns=['iata_code_Origin', 'iata_code_Dest'])

# step 1 is to get correct and reliable datetime objects in UTC and Local time for the departure and arrival times.
# we start with the departure time where we combine the filght date datetime with the daparture time timeobject to get a datetime object in local time
dataset['CRSDepDateTime'] = pd.to_datetime(dataset['FlightDate'] + ' ' + dataset['CRSDepTime'].astype(str), errors='coerce')
dataset['CRSDepDateTime_UTC'] = dataset.groupby('TZ_Origin')['CRSDepDateTime'].transform(
    lambda x: x.dt.tz_localize(x.name, ambiguous=True,nonexistent='shift_forward').dt.tz_convert('UTC')
)
# we convert the departure time to UTC time, by using the timezone of the departure airport

# we do the same for the arrival time
dataset['CRSArrDateTime'] = pd.to_datetime(dataset['FlightDate'] + ' ' + dataset['CRSArrTime'].astype(str), errors='coerce')
dataset['CRSArrDateTime_UTC'] = dataset.groupby('TZ_Dest')['CRSArrDateTime'].transform(
    lambda x: x.dt.tz_localize(x.name,ambiguous=True,nonexistent='shift_forward').dt.tz_convert('UTC')
)
# also convert the actual arrival and departure times to UTC time, by using the timezone of the departure and arrival airport
# check if the DepTime is smaller than the CRSDepTime, if it is, we add 1 day to the DepTime, because it means that the flight departed after midnight
mask = ((dataset['DepTime'] < dataset['CRSDepTime'] ) & (dataset['DepDelay'] > 0)).astype(int)

dataset['DepDateTime'] = pd.to_datetime(dataset['FlightDate'] + ' ' + dataset['DepTime'].astype(str), errors='coerce')
# add 1 day to the DepDateTime if the DepTime is smaller than the CRSDepTime, because it means that the flight departed after midnight
dataset['DepDateTime'] = dataset['DepDateTime'] + pd.to_timedelta(mask, unit='D')

dataset['DepDateTime_UTC'] = dataset.groupby('TZ_Origin')['DepDateTime'].transform(
    lambda x: x.dt.tz_localize(x.name, ambiguous=True,nonexistent='shift_forward').dt.tz_convert('UTC')
)
dataset['ArrDateTime'] = pd.to_datetime(dataset['FlightDate'] + ' ' + dataset['ArrTime'].astype(str), errors='coerce')
dataset['ArrDateTime_UTC'] = dataset.groupby('TZ_Dest')['ArrDateTime'].transform(
    lambda x: x.dt.tz_localize(x.name, ambiguous=True,nonexistent='shift_forward').dt.tz_convert('UTC')
)


# we check if the arrival time is before the departure time, if it is, we add 1 day to the arrival time
# we get a boolean mask where the arrival time is before the departure time
mask = (dataset['CRSArrDateTime_UTC'] < dataset['CRSDepDateTime_UTC']).astype(int)
dataset['CRSArrDateTime_UTC'] = dataset['CRSArrDateTime_UTC'] + pd.to_timedelta(mask, unit='D')
dataset['CRSArrDateTime'] = dataset['CRSArrDateTime'] + pd.to_timedelta(mask, unit='D')

mask = (dataset['ArrDateTime_UTC'] < dataset['DepDateTime_UTC']).astype(int)
dataset['ArrDateTime_UTC'] = dataset['ArrDateTime_UTC'] + pd.to_timedelta(mask, unit='D')
dataset['ArrDateTime'] = dataset['ArrDateTime'] + pd.to_timedelta(mask, unit='D')

display(dataset[['FlightDate','CRSDepTime', 'CRSDepDateTime', 'CRSDepDateTime_UTC', 'CRSArrTime', 'CRSArrDateTime_UTC']].head())



In [ ]:
# make a temporary Collumn for k hours before departure
hours_before = 2
dataset["knownWeatherDateTime_UTC"] = dataset["CRSDepDateTime_UTC"] - pd.to_timedelta(hours_before, unit='h')
# round down to the previus full hour
dataset["knownWeatherDateTime_UTC"] = dataset["knownWeatherDateTime_UTC"].dt.floor('h')
# remove the timezone information, as the weather data does not have timezone information
dataset["knownWeatherDateTime_UTC"] = dataset["knownWeatherDateTime_UTC"].dt.tz_localize(None)

# merge with weather data on ORIGIN and knownWeatherDateTime_UTC

display(dataset)

In [ ]:
# save the cleaned data to a new csv file
# save the cleaned data to a new parquet file
# save the cleaned data to a new parquet file (write directly to S3 with storage options)
target = f"s3://data-mining/{Destination_folder}{output_file}"
dataset.to_parquet(target, index=False, storage_options=storage.storage_options)